# Tensorflow

In [1]:
import tensorflow as tf

In [2]:
mnist = tf.keras.datasets.mnist

# 載入 MNIST 手寫阿拉伯數字資料
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# 特徵縮放，使用常態化(Normalization)，公式 = (x - min) / (max - min)
x_train_norm, x_test_norm = x_train / 255.0, x_test / 255.0

# 轉為 Dataset，含 X/Y 資料
train_ds = tf.data.Dataset.from_tensor_slices((x_train_norm, y_train))

In [3]:
# 建立模型
model = tf.keras.models.Sequential(
    [
        tf.keras.layers.Flatten(input_shape=(28, 28)),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(10, activation='softmax'),
    ]
)

/Users/rikenmi/Work/study/python/PyTorch_Book/.venv/lib/python3.12/site-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [4]:
# 設定優化器(optimizer)、損失函數(loss)、效能衡量指標(metrics)的類別
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

## 或是 one-hot encoding

In [5]:
# 將 training 的 label 進行 one-hot encoding，例如數字 7 經過 One-hot encoding 轉換後是 0000000100，即第8個值為 1
y_train = tf.keras.utils.to_categorical(y_train)
y_test = tf.keras.utils.to_categorical(y_test)
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

## 模型訓練

In [6]:
model.fit(x_train_norm, y_train, epochs=5, validation_split=0.2)

Epoch 1/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.9045 - loss: 0.3282 - val_accuracy: 0.9563 - val_loss: 0.1555
Epoch 2/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 1s 928us/step - accuracy: 0.9536 - loss: 0.1580 - val_accuracy: 0.9668 - val_loss: 0.1131
Epoch 3/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 1s 931us/step - accuracy: 0.9654 - loss: 0.1172 - val_accuracy: 0.9737 - val_loss: 0.0951
Epoch 4/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.9709 - loss: 0.0937 - val_accuracy: 0.9732 - val_loss: 0.0906
Epoch 5/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 1s 893us/step - accuracy: 0.9755 - loss: 0.0793 - val_accuracy: 0.9741 - val_loss: 0.0857


In [7]:
# 評分(Score Model)
score = model.evaluate(x_test_norm, y_test, verbose=0)

for i, x in enumerate(score):
    print(f'{model.metrics_names[i]}: {score[i]:.4f}')

loss: 0.0784
compile_metrics: 0.9749


# PyTorch

In [8]:
import torch
from torch import nn, optim
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import MNIST

In [9]:
PATH_DATASETS = "data"  # 預設路徑
# 下載 MNIST 手寫阿拉伯數字 訓練資料
train_ds = MNIST(PATH_DATASETS, train=True, download=True, transform=transforms.ToTensor())

# 下載測試資料
test_ds = MNIST(PATH_DATASETS, train=False, download=True, transform=transforms.ToTensor())

In [10]:
device = "cuda" if torch.cuda.is_available() else "mps" if torch.mps.is_available() else "cpu"
device

'mps'

In [11]:
# 建立模型
model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(28 * 28, 256),
    nn.Dropout(0.2),
    nn.Linear(256, 10),
    # 使用nn.CrossEntropyLoss()時，不需要將輸出經過softmax層，否則計算的損失會有誤
    # nn.Softmax(dim=1),
).to(device)

## 模型訓練

In [12]:
epochs = 5
lr = 0.1
BATCH_SIZE = 1024  # 批量

# 建立 DataLoader
train_loader = DataLoader(train_ds, batch_size=600)

# 設定優化器(optimizer)
# optimizer = optim.Adam(model.parameters(), lr=lr)
optimizer = optim.Adadelta(model.parameters(), lr=lr)

criterion = nn.CrossEntropyLoss()

In [13]:
model.train()
loss_list = []
for epoch in range(1, epochs + 1):
    data: torch.Tensor
    target: torch.Tensor
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        #         if batch_idx == 0 and epoch == 1: print(data[0])

        optimizer.zero_grad()
        output: torch.Tensor = model(data)
        loss = criterion.forward(output, target)
        loss.backward()
        optimizer.step()

        if batch_idx % 10 == 0:
            loss_list.append(loss.item())
            batch = (batch_idx + 10) * len(data)
            data_count = len(cast(Sized, train_loader.dataset))
            percentage = 100.0 * (batch_idx + 10) / data_count
            print(f'Epoch {epoch}: [{batch:5d} / {data_count}] ({percentage:.0f} %)  Loss: {loss.item():.6f}')

Epoch 1: [ 6000 / 100] (10 %)  Loss: 2.330796
Epoch 1: [12000 / 100] (20 %)  Loss: 2.045637
Epoch 1: [18000 / 100] (30 %)  Loss: 1.826451
Epoch 1: [24000 / 100] (40 %)  Loss: 1.586127
Epoch 1: [30000 / 100] (50 %)  Loss: 1.366418
Epoch 1: [36000 / 100] (60 %)  Loss: 1.244214
Epoch 1: [42000 / 100] (70 %)  Loss: 1.006315
Epoch 1: [48000 / 100] (80 %)  Loss: 0.989476
Epoch 1: [54000 / 100] (90 %)  Loss: 0.722534
Epoch 1: [60000 / 100] (100 %)  Loss: 0.735492
Epoch 2: [ 6000 / 100] (10 %)  Loss: 0.701793
Epoch 2: [12000 / 100] (20 %)  Loss: 0.552623
Epoch 2: [18000 / 100] (30 %)  Loss: 0.653311
Epoch 2: [24000 / 100] (40 %)  Loss: 0.552205
Epoch 2: [30000 / 100] (50 %)  Loss: 0.581955
Epoch 2: [36000 / 100] (60 %)  Loss: 0.627462
Epoch 2: [42000 / 100] (70 %)  Loss: 0.509458
Epoch 2: [48000 / 100] (80 %)  Loss: 0.611092
Epoch 2: [54000 / 100] (90 %)  Loss: 0.415893
Epoch 2: [60000 / 100] (100 %)  Loss: 0.470293
Epoch 3: [ 6000 / 100] (10 %)  Loss: 0.469931
Epoch 3: [12000 / 100] (20 %)  L

In [ ]:
# 建立 DataLoader
test_loader = DataLoader(test_ds, shuffle=False, batch_size=BATCH_SIZE)

model.eval()
test_loss = 0
correct = 0
with torch.no_grad():
    data: torch.Tensor
    target: torch.Tensor
    for data, target in test_loader:
        data, target = data.to(device), target.to(device)
        output: torch.Tensor = model(data)

        # sum up batch loss
        test_loss += criterion.forward(output, target).item()

        # 預測
        pred = output.argmax(dim=1, keepdim=True)

        # 正確筆數
        correct += pred.eq(target.view_as(pred)).sum().item()

# 平均損失
test_loss /= len(cast(Sized, test_loader.dataset))
# 顯示測試結果
batch = batch_idx * len(data)
data_count = len(cast(Sized, test_loader.dataset))
percentage = 100.0 * correct / data_count
print(f'平均損失: {test_loss:.4f}, 準確率: {correct}/{data_count} ({percentage:.0f}%)\n')

平均損失: 0.0003, 準確率: 9058/10000 (91%)

